# Assignment 2 – Dynamic Chunking + FAISS Vector Database

This notebook uses the three  files:
- `artificial_intelligence.txt`
- `data_warehousing.txt`
- `world_foods.txt`

Phase 1 runs only when the saved FAISS database does not exist. Later runs load the saved index and metadata directly.


In [1]:
!pip -q install sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.5 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuration and File Upload


In [14]:
import os, json, re
import numpy as np
import faiss
from google.colab import files
from sentence_transformers import SentenceTransformer

import os

BASE_PATH = "/content/drive/MyDrive/Assignment_2_VectorDB"

FILE_NAMES = [
    f"{BASE_PATH}/artificial_intelligence.txt",
    f"{BASE_PATH}/data_warehousing.txt",
    f"{BASE_PATH}/world_foods.txt"
]

INDEX_FILE = f"{BASE_PATH}/index.faiss"
METADATA_FILE = f"{BASE_PATH}/metadata.json"
MODEL_NAME = 'all-MiniLM-L6-v2'

missing = [f for f in FILE_NAMES if not os.path.exists(f)]
if missing:
    print('Please upload these files:', missing)
    files.upload()

model = SentenceTransformer(MODEL_NAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Phase 1 – Ingestion (Runs Once)
Dynamic chunking groups complete paragraphs until a target size is reached, rather than cutting at a rigid character boundary.


In [15]:
def dynamic_chunking(text, target_size=700):
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    chunks, current = [], ''
    for paragraph in paragraphs:
        candidate = paragraph if not current else current + '\n\n' + paragraph
        if len(candidate) <= target_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = paragraph
    if current:
        chunks.append(current)
    return chunks

if not (os.path.exists(INDEX_FILE) and os.path.exists(METADATA_FILE)):
    print('Phase 1: Creating embeddings and FAISS vector database...')
    metadata = []
    all_chunks = []

    for file_name in FILE_NAMES:
        with open(file_name, 'r', encoding='utf-8') as f:
            text = f.read()
        file_chunks = dynamic_chunking(text)
        print(f'{file_name}: {len(file_chunks)} chunks')
        for chunk_number, chunk in enumerate(file_chunks, 1):
            all_chunks.append(chunk)
            metadata.append({
                'source': file_name,
                'chunk_number': chunk_number,
                'text': chunk
            })

    embeddings = model.encode(all_chunks, convert_to_numpy=True).astype('float32')
    # L2-normalized vectors + inner product = cosine similarity
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, INDEX_FILE)
    with open(METADATA_FILE, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f'Phase 1 completed. Stored {index.ntotal} vectors.')
else:
    print('Phase 1 skipped: saved vector database already exists.')


Phase 1: Creating embeddings and FAISS vector database...
/content/drive/MyDrive/Assignment_2_VectorDB/artificial_intelligence.txt: 4 chunks
/content/drive/MyDrive/Assignment_2_VectorDB/data_warehousing.txt: 4 chunks
/content/drive/MyDrive/Assignment_2_VectorDB/world_foods.txt: 4 chunks
Phase 1 completed. Stored 12 vectors.


## Phase 2 – Load Saved Vector DB and Search
No document chunking or document embedding is repeated here.


In [16]:
index = faiss.read_index(INDEX_FILE)
with open(METADATA_FILE, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print('Loaded vectors:', index.ntotal)

def search(question, top_k=3):
    query_vector = model.encode([question], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_vector)
    scores, indices = index.search(query_vector, top_k)

    results = []
    print('\nQuestion:', question)
    print('\nTop 3 Answers:')
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        item = metadata[idx]
        result = {**item, 'score': float(score)}
        results.append(result)
        print(f'\nAnswer {rank}')
        print(f'Similarity Score: {score:.4f}')
        print(f'Reference: {item["source"]}')
        print(f'Chunk: {item["chunk_number"]}')
        print(item['text'])
    return results


Loaded vectors: 12


## Ask Questions Repeatedly


In [ ]:
while True:
    question = input('\nEnter your question (q to quit): ').strip()
    if question.lower() in {'q', 'quit', 'exit'}:
        print('Search ended.')
        break
    if not question:
        print('Please enter a question.')
        continue
    results = search(question, top_k=3)



Enter your question (q to quit): What is the difference between OLTP and OLAP?

Question: What is the difference between OLTP and OLAP?

Top 3 Answers:

Answer 1
Similarity Score: 0.6080
Reference: /content/drive/MyDrive/Assignment_2_VectorDB/data_warehousing.txt
Chunk: 1
A data warehouse combines data from operational systems for reporting and analysis. It often stores historical information and organizes data consistently across business areas. Analysts use it to study trends and support decisions.

OLTP systems support operational transactions such as recording payments. OLAP systems support analytical queries across large datasets. Operational databases favor fast individual updates, while warehouses favor aggregation and historical analysis.

ETL stands for extract, transform, and load. It extracts data from sources, cleans and transforms it, then loads it into a target. ELT loads data first and performs transformations inside the target platform.

Answer 2
Similarity Score: 0.25

### Example questions
- `What is the difference between OLTP and OLAP?`
- `How does RAG retrieve information?`
- `What ingredients are used in Kabsa?`
